<a href="https://colab.research.google.com/github/renadmbd/Recommender-Systems/blob/main/RS4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 1 — Warm-Up Tutorial: Item-Based Recommender

In [1]:
#Step 1 - Import the libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import random

#Step 2 - Load the dataset into a dataframe
ratings = pd.read_csv("ccai422_lab04_part1_data.csv", index_col='Unnamed: 0')
print(ratings.head(5))

#Step 3 - Explore the data
print('The number of data points in this dataset: ' + str(len(ratings)))
print('The number of items (i.e. movies) in the dataset: ' + str(ratings['movie_id'].nunique()))
print('The number of users in the dataset: ' + str(ratings['user_id'].nunique()))

ratings_per_users = ratings.groupby('user_id').count()
print('The average ratings per user: ' + str(round(ratings_per_users.mean().iloc[0], 2)))
print('The below table shows the number of ratings per user\n')
print(ratings_per_users)

#Step 4 - Build the rating matrix
ratings = ratings.pivot_table(values='rating', index='user_id', columns='movie_id')
cos_matrix_dummy = ratings.copy()
cos_matrix_dummy = cos_matrix_dummy.rename_axis('user_id', axis=1).rename_axis(None, axis=0)

#Step 5 - Center the data around the mean
cos_matrix_dummy['mean'] = cos_matrix_dummy.mean(axis=1)
cos_matrix_dummy.loc[:, cos_matrix_dummy.columns != 'mean'] = (
    cos_matrix_dummy.loc[:, cos_matrix_dummy.columns != 'mean']
    - cos_matrix_dummy['mean'].values[:, None]
)
cos_matrix_dummy.drop(columns='mean', inplace=True)
print(cos_matrix_dummy.head())

#Step 6 - Compute cosine similarity
cos_matrix_dummy.fillna(0, inplace=True)
cos_matrix_dummy_sim = cosine_similarity(cos_matrix_dummy.T)
cos_matrix_dummy_sim = pd.DataFrame(
    cos_matrix_dummy_sim,
    columns=list(cos_matrix_dummy.T.index),
    index=list(cos_matrix_dummy.T.index)
)
print(cos_matrix_dummy_sim)

#Step 7 - Select the ratings data for a random user
random.seed(3)
random_user = random.randrange(len(ratings))
print('random user ID is {}'.format(random_user))
random_user_ratings = ratings[ratings.index == random_user]

#Step 8 - Randomly select an unrated item for that user
unrated_items_for_random_user = random_user_ratings.columns[random_user_ratings.isnull().all(0)]
random_unrated_item = random.choice(list(unrated_items_for_random_user))
print('Item {} is unrated by user {}'.format(random_unrated_item, random_user))

#Step 9 - Select the top-n similar items to the unrated item
rated_items_for_random_user = random_user_ratings.columns[random_user_ratings.notnull().all(0)]
filtered_col_sim = list(rated_items_for_random_user)
filtered_col_sim.append(random_unrated_item)

rated_unrated_sim_random_user = cos_matrix_dummy_sim.loc[
    cos_matrix_dummy_sim.columns.isin(filtered_col_sim)
]

topn = rated_unrated_sim_random_user.nlargest(3, random_unrated_item).index.tolist()[1:]
neighbors_item_ratings_random_item = random_user_ratings.loc[:, topn]
neighbors_sim = rated_unrated_sim_random_user[[random_unrated_item]].nlargest(3, random_unrated_item)[1:]

print("The top neighbors for the item {} that is unrated by user {} are the items: {}".format(
    random_unrated_item, random_user, topn))
print("The ratings for the top {} neighbors items are: {}".format(
    len(topn), *neighbors_item_ratings_random_item.values))
print("The cosine similarities between item {} and items: {} are: {}".format(
    random_unrated_item, topn, neighbors_sim.values))

#Step 10 - Predict the rating of the randomly selected item
predicted_value_for_unrated_item = (
    neighbors_sim.T.dot(neighbors_item_ratings_random_item.T).values[0]
    / neighbors_sim.sum()
)
print("Predicted rating:", predicted_value_for_unrated_item)


   user_id  movie_id  rating
0      196       242       3
1      186       302       3
2       22       377       1
3      244        51       2
4      166       346       1
The number of data points in this dataset: 100000
The number of items (i.e. movies) in the dataset: 1682
The number of users in the dataset: 943
The average ratings per user: 106.04
The below table shows the number of ratings per user

         movie_id  rating
user_id                  
1             272     272
2              62      62
3              54      54
4              24      24
5             175     175
...           ...     ...
939            49      49
940           107     107
941            22      22
942            79      79
943           168     168

[943 rows x 2 columns]
user_id      1         2         3         4         5         6         7     \
1        1.389706 -0.610294  0.389706 -0.610294 -0.610294  1.389706  0.389706   
2        0.290323       NaN       NaN       NaN       NaN       Na

Part 2 — Build an Item-Based Recommender with sklearn

In [2]:
#Step 1 - Import the libraries
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

#Step 2 - Load the dataset into a dataframe
ratings = pd.read_csv("ccai422_lab04_part2_3_data.csv", index_col=0)
print(ratings)

#Step 3 - Apply plain (non-adjusted) cosine similarity with sklearn
ratings_filled = ratings.fillna(0)

item_sim = cosine_similarity(ratings_filled)
item_sim_df = pd.DataFrame(
    item_sim,
    index=ratings_filled.index,
    columns=ratings_filled.index
)
print("Item-item cosine similarity matrix:")
print(item_sim_df)

#Step 4 - Predict the rating for Item5 (Alice's missing rating)
target_user = "Alice"
target_item = "item5"

# Items Alice has actually rated
rated_items = ratings[target_user].dropna().index.tolist()

# Similarities between item5 and the items Alice rated
sims_to_target = item_sim_df.loc[rated_items, target_item]

# Top 2 most similar (neighbor) items
top2_neighbors = sims_to_target.nlargest(2)
print("Top 2 neighbors of {}: \n{}".format(target_item, top2_neighbors))

# Alice's ratings for those neighbor items
neighbor_ratings = ratings.loc[top2_neighbors.index, target_user]

# Prediction formula: weighted average using similarity as weights
predicted_rating = (top2_neighbors.values * neighbor_ratings.values).sum() / top2_neighbors.values.sum()

print("Predicted rating for {}'s {}: {:.4f}".format(target_user, target_item, predicted_rating))


       Alice  User1  User2  User3  User4
item1    5.0      3      4      3      1
item2    3.0      1      3      3      5
item3    4.0      2      4      1      5
item4    4.0      3      3      5      2
item5    NaN      3      5      4      1
Item-item cosine similarity matrix:
          item1     item2     item3     item4     item5
item1  1.000000  0.780260  0.819782  0.943370  0.759257
item2  0.780260  1.000000  0.942020  0.847984  0.673201
item3  0.819782  0.942020  1.000000  0.784025  0.622425
item4  0.943370  0.847984  0.784025  1.000000  0.811526
item5  0.759257  0.673201  0.622425  0.811526  1.000000
Top 2 neighbors of item5: 
item4    0.811526
item1    0.759257
Name: item5, dtype: float64
Predicted rating for Alice's item5: 4.4834


Part 3 — Individual Assessment: Cosine Similarity from Scratch

In [5]:
#Step 1 - Import the libraries
import pandas as pd
import numpy as np

#Step 2 - Load the dataset into a dataframe
ratings = pd.read_csv("ccai422_lab04_part2_3_data.csv", index_col=0)
print(ratings)

#Step 3 - Implement cosine similarity from scratch
def cosine_similarity_manual(vec_a, vec_b):
    """
    Computes cosine similarity between two vectors using only ratings
    where BOTH items were rated by the same user (co-rated entries),
    matching how sklearn treats the NaN-filled-with-0 columns.
    """
    vec_a = np.array(vec_a, dtype=float)
    vec_b = np.array(vec_b, dtype=float)

    dot_product = np.sum(vec_a * vec_b)
    norm_a = np.sqrt(np.sum(vec_a ** 2))
    norm_b = np.sqrt(np.sum(vec_b ** 2))

    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

#Step 4 - Build the item-item similarity matrix manually
ratings_filled = ratings.fillna(0)
items = ratings_filled.index
n_items = len(items)

sim_matrix = np.zeros((n_items, n_items))

for i in range(n_items):
    for j in range(n_items):
        vec_a = ratings_filled.iloc[i, :]
        vec_b = ratings_filled.iloc[j, :]
        sim_matrix[i, j] = cosine_similarity_manual(vec_a, vec_b)

item_sim_df = pd.DataFrame(sim_matrix, index=items, columns=items)
print("Manual item-item cosine similarity matrix:")
print(item_sim_df)

#Step 5 - Predict Alice's Item5 rating using the same prediction formula
target_user = "Alice"
target_item = "item5"

rated_items = ratings[target_user].dropna().index.tolist()
sims_to_target = item_sim_df.loc[rated_items, target_item]

top2_neighbors = sims_to_target.nlargest(2)
neighbor_ratings = ratings.loc[top2_neighbors.index, target_user]

predicted_rating_manual = (top2_neighbors.values * neighbor_ratings.values).sum() / top2_neighbors.values.sum()

print("Top 2 neighbors of {}: \n{}".format(target_item, top2_neighbors))
print("Predicted rating (manual) for {}'s {}: {:.4f}".format(target_user, target_item, predicted_rating_manual))

#Step 6 - Confirm the result matches Part 2
print("Matches Part 2 prediction:", round(predicted_rating_manual, 4) == 4.4834)


       Alice  User1  User2  User3  User4
item1    5.0      3      4      3      1
item2    3.0      1      3      3      5
item3    4.0      2      4      1      5
item4    4.0      3      3      5      2
item5    NaN      3      5      4      1
Manual item-item cosine similarity matrix:
          item1     item2     item3     item4     item5
item1  1.000000  0.780260  0.819782  0.943370  0.759257
item2  0.780260  1.000000  0.942020  0.847984  0.673201
item3  0.819782  0.942020  1.000000  0.784025  0.622425
item4  0.943370  0.847984  0.784025  1.000000  0.811526
item5  0.759257  0.673201  0.622425  0.811526  1.000000
Top 2 neighbors of item5: 
item4    0.811526
item1    0.759257
Name: item5, dtype: float64
Predicted rating (manual) for Alice's item5: 4.4834
Matches Part 2 prediction: True
